# Laboratorio 5. Formatos de datos

Uso de `csv`, `json`, `configparser` y `tomllib` para trabajar con formatos habituales sin paquetes externos.

## 1. Preparar rutas del tema

In [ ]:
from pathlib import Path


def localizar_tema10() -> Path:
    """Localiza el directorio Tema10 aunque el notebook se ejecute desde notebooks, Tema10 o Curso_Python."""
    actual = Path.cwd().resolve()
    candidatos = [
        actual,
        actual.parent,
        actual / "Tema10",
        actual.parent / "Tema10",
    ]

    for candidato in candidatos:
        if candidato.name == "Tema10" and candidato.exists():
            return candidato
        if (candidato / "src").exists() and (candidato / "notebooks").exists():
            return candidato

    raise RuntimeError("No se ha podido localizar el directorio Tema10. Abre el notebook desde Tema10/notebooks o desde el proyecto Curso_Python.")


BASE_DIR = localizar_tema10()
SRC_DIR = BASE_DIR / "src"
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"
LOG_DIR = BASE_DIR / "logs"

for directorio in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR, LOG_DIR):
    directorio.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)

## 2. Importar módulos

In [ ]:
import csv
import json
import configparser
import tomllib
from pathlib import Path

## 3. CSV: escribir y leer datos tabulares

In [ ]:
ruta_csv = DATA_DIR / "servicios.csv"
filas = [
    {"servicio": "ssh", "puerto": 22, "estado": "OK"},
    {"servicio": "nginx", "puerto": 443, "estado": "OK"},
    {"servicio": "api", "puerto": 8080, "estado": "ERROR"},
]

with open(ruta_csv, "w", newline="", encoding="utf-8") as fichero:
    escritor = csv.DictWriter(fichero, fieldnames=["servicio", "puerto", "estado"])
    escritor.writeheader()
    escritor.writerows(filas)

with open(ruta_csv, "r", newline="", encoding="utf-8") as fichero:
    lector = csv.DictReader(fichero)
    for fila in lector:
        print(f"{fila['servicio']} -> {fila['puerto']} [{fila['estado']}]")

## 4. JSON: escribir y leer datos estructurados

In [ ]:
ruta_json = DATA_DIR / "servicios.json"
ruta_json.write_text(json.dumps(filas, indent=4, ensure_ascii=False), encoding="utf-8")

servicios = json.loads(ruta_json.read_text(encoding="utf-8"))
print("Servicios cargados desde JSON:", len(servicios))
print("Primer servicio:", servicios[0])

## 5. INI con `configparser`

El notebook usa `Tema10/data/database.ini` si existe. Si no existe, lo crea para que el laboratorio sea reproducible.

In [ ]:
ruta_ini = DATA_DIR / "database.ini"
if not ruta_ini.exists():
    ruta_ini.write_text(
        "[mysqld]"
        "host = localhost"
        "port = 3306"
        "user = soporte",
        encoding="utf-8",
    )

config_ini = configparser.ConfigParser()
config_ini.read(ruta_ini, encoding="utf-8")
print("Host extraído de INI:", config_ini["mysqld"]["host"])
print("Puerto extraído de INI:", config_ini["mysqld"]["port"])

## 6. TOML con `tomllib`

`tomllib` lee TOML en modo binario (`rb`) y conserva tipos como enteros y booleanos.

In [ ]:
ruta_toml = DATA_DIR / "app_config.toml"
if not ruta_toml.exists():
    ruta_toml.write_text(
        "[server]"
        "host = '127.0.0.1'"
        "port = 8000"
        "debug_mode = true",
        encoding="utf-8",
    )

with open(ruta_toml, "rb") as fichero:
    config_toml = tomllib.load(fichero)

print("Host TOML:", config_toml["server"]["host"])
print("Puerto TOML:", config_toml["server"]["port"], type(config_toml["server"]["port"]))
print("Modo debug de TOML:", config_toml["server"]["debug_mode"], type(config_toml["server"]["debug_mode"]))